# 진짜 RAG 챗봇 완성하기

**위니브 카페 CS 챗봇 만들기 **


## 오늘 배울 것 

> 1~3교시 부품을 **하나로 연결**합니다. 질문하면 → 알아서
> 검색하고 → 그 문서를 보고 → **출처까지 붙여** 답하는 진짜
> 챗봇을 완성합니다.

| 단계 | 내용 |
|------|------|
| 1️⃣ 이론 | 흩어진 부품을 잇는다 + 검색기란? |
| 2️⃣ 기본 | 검색기 만들기 + 검색 결과 정리하기 |
| 3️⃣ 응용 | 검색 → AI 를 한 줄로 연결하기 |
| 4️⃣ 실습 | 위니브 RAG 챗봇 완성 (출처 + 거짓말 방지) |


# 1️⃣ 단계 — 이론 

## 지금까지 만든 부품들

| 순서 | 만든 것 |
|------|---------|
| 1 | 문서를 같이 주면 AI가 거짓말 안 한다 (RAG의 핵심) |
| 2 | 문서를 작은 청크로 쪼갰다 |
| 3 | 청크를 의미로 검색되게 벡터 DB에 저장했다 |

부품은 다 있는데 **따로 놀고 있습니다.** 어제는 검색을 직접 손으로
프롬프트에 붙였습니다.
사람이 매번 손으로 잇는 건 챗봇이 아닙니다.

> 🗝️ **오늘 할 일**: "질문 → 검색 → 답변" 을 자동으로 한 번에
> 흐르게 연결한다. 그러면 진짜 RAG 챗봇이 됩니다.

## 새 부품: 검색기(Retriever)

어제는 `db.similarity_search("질문", k=3)` 으로 직접
검색했습니다. 잘 됐지만, 자동 연결하기엔 좀 번거롭습니다.

그래서 벡터 DB를 **"질문 넣으면 청크 나오는 표준 창구"** 로
한 번 감쌉니다. 이게 **검색기(Retriever)** 입니다.

> 💡 비유:
> - 벡터 DB = 책이 정리된 **서고**
> - 검색기 = 서고 앞의 **사서**
>
> 우리는 서고에 직접 안 들어가고, 사서에게 "이거 관련 자료
> 주세요" 한마디만 하면 됩니다. 자동 연결엔 '사서'가 편합니다.

## RAG 챗봇의 전체 흐름

```
① 고객 질문         "개봉한 원두 환불돼요?"
      ↓
② 검색기가 검색      관련 청크 3개를 벡터 DB에서 꺼냄
      ↓
③ 청크 정리          꺼낸 청크들을 하나의 글로 합침 (+출처)
      ↓
④ AI에게 전달        "이 자료만 보고 답해.  자료=③  질문=①"
      ↓
⑤ AI가 답 생성       "개봉 원두는 환불 불가. (출처: 환불정책)"
```

어제 1교시에서는 "책 전체를 줬다"면, 오늘은 ②검색이 "질문에 맞는 부분만"
자동으로 찾아줍니다. 

## 고객센터 챗봇의 필수 예의 — 출처 + 거짓말 방지

답이 맞아도 **근거를 못 대면** 고객은 안 믿습니다.

> 🤖 (출처 없음) "환불 불가합니다." → "왜요?" 불신
> 🤖 (출처 있음) "환불 불가합니다. **(출처: 환불정책)**" → 신뢰

출처는 2교시에서 청크에 달아둔 꼬리표에서 옵니다. 그리고 1교시의
안전장치("문서에 없으면 모른다고 해")도 같이 넣어, 검색이 엉뚱한
걸 가져온 경우의 거짓말도 한 번 더 막습니다.

---
👉 아래부터 코드입니다. 한 줄씩 천천히 이해해 봅시다.


# 2️⃣ 단계 — 검색기 만들고 결과 정리하기

작은 예제로 "검색기"와 "청크 정리"가 뭔지 쉬운 형태로 봅니다.

### 셀 1 — 도구 설치

In [1]:
!pip install -q langchain langchain-openai langchain-community langchain-chroma langchain-text-splitters langchain-huggingface python-dotenv sentence-transformers

### 셀 2 — 키 넣고 임베딩 준비

OpenAI 키를 읽어옵니다. 임베딩은 키 있으면 OpenAI, 없으면
무료 모델로 자동 선택됩니다. (답변 생성은 OpenAI 키 필요)

In [2]:
import os

try:
    from google.colab import userdata
    os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
  
except Exception:
    from dotenv import load_dotenv
    load_dotenv()


has_openai = bool(os.environ.get("OPENAI_API_KEY"))

if has_openai:
    from langchain_openai import OpenAIEmbeddings
    embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
    print("✅ OpenAI 키 OK — 답변 생성까지 가능합니다")
else:
    from langchain_huggingface import HuggingFaceEmbeddings
    embeddings = HuggingFaceEmbeddings(
        model_name="sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
    )
    print("⚠️ OpenAI 키 없음 — 검색까지는 되지만, 답변 생성 셀은 키가 필요합니다")

✅ OpenAI 키 OK — 답변 생성까지 가능합니다


### 셀 3 — 연습용 벡터 DB 만들기

작은 문장 3개로 벡터 DB를 만듭니다.

In [21]:
#벡터 DB 패키지 임포트  
from langchain_chroma import Chroma
from langchain_core.documents import Document

# 아래 3가지 내용을 도큐먼트로 저장한 후 sample_docs 리스트로 저장하세요

sample_docs = [
    Document(page_content="환불은 수령 후 7일 이내 신청 가능합니다.", metadata={"source": "환불정책"}),
    Document(page_content="배송비는 3,000원, 5만원 이상 무료입니다.", metadata={"source": "배송정책"}),
    Document(page_content="회원 등급은 누적 구매액으로 결정됩니다.", metadata={"source": "멤버십안내"}),
]

db = Chroma.from_documents(
    sample_docs, embeddings,
    collection_name="practice",
)

    # 내용 : "환불은 수령 후 7일 이내 신청 가능합니다."   출처 : "환불정책"
    # 내용 :"배송비는 3,000원, 5만원 이상 무료입니다."  출처 :"배송정책"
    # 내용 :"회원 등급은 누적 구매액으로 결정됩니다."    출처 :"멤버십안내"

# db = sample_docs 를 Chroma 벡터 데이터베이스에 이름은 "practice"로 저장하세요

print("연습용 벡터 DB 준비 완료 ✅")

연습용 벡터 DB 준비 완료 ✅


**이 셀이 한 일**: 연습용 문장 3개를 벡터 DB에 넣었습니다.


### 셀 4 — 벡터 DB를 '검색기'로 감싸기

`db.similarity_search(...)` 를 직접 썼죠. 이번엔
`as_retriever()` 로 **검색 창구(사서)** 를 만듭니다.

In [6]:
# 벡터 DB → 검색기 (가까운 청크 2개를 주도록 설정)
retriever = db.as_retriever(search_kwargs={"k":2})

# 검색기는 "질문" "돈 언제 돌려받아요?" 만 넣으면 청크 리스트 받습니다.
found = retriever.invoke("돈 언제 돌려받아요?")

print("찾은 청크 수:", len(found))
# 아래 출력 예를 보고 찾은 청크를 모두 출력하세요

for doc in found:
    print(" _",doc.page_content, "|출처|", doc.metadata["source"])

찾은 청크 수: 2
 _ 회원 등급은 누적 구매액으로 결정됩니다. |출처| 멤버십안내
 _ 배송비는 3,000원, 5만원 이상 무료입니다. |출처| 배송정책


**이 셀이 한 일**: 벡터 DB를 검색기로 감쌌습니다.

> 💡 `similarity_search` 와 결과는 같지만, `retriever.invoke(질문)`
> 형태라 다음 단계에서 **자동 연결**하기 좋습니다.

### 셀 5 — 검색된 청크들을 하나의 글로 정리

검색 결과는 청크 *여러 개* 입니다. AI에게 주려면 하나의 글로
합쳐야 합니다. 출처도 같이 묶습니다.

In [9]:
def format_docs(docs):
    # 각 청크를 "[출처: ...] 본문" 형태로 만들고, 빈 줄로 이어붙임 기능을 완성하세요
    blocks = []
    for doc  in docs:
        blocks.append(f"[출처:{doc.metadata['source']}]\n{doc.page_content}")
    return "\n\n".join(blocks)

# 검색 → 정리 해보기
docs =  retriever.invoke("환불 언제까지 신청 가능 한가요?")
print(format_docs(docs))

[출처:환불정책]
환불은 수령 후 7일 이내 신청 가능합니다.

[출처:배송정책]
배송비는 3,000원, 5만원 이상 무료입니다.


**이 셀이 한 일**: 검색된 청크 여러 개를 출처가 붙은 **하나의
글**로 합쳤습니다.

> 📌 정리: 벡터 DB → `as_retriever()` 로 검색기 만들기 →
> `format_docs` 로 결과를 한 글로 정리. 이게 RAG의 재료 준비입니다.

# 3️⃣ 단계 — 검색 → AI 를 한 줄로 연결

이제 "검색"과 "AI 답변"을 자동으로 잇습니다. 1교시에서 본
`프롬프트 | AI` 앞에 **검색 단계**를 붙이는 게 전부입니다.

### 셀 6 — 검색 결과를 넣을 프롬프트 만들기

 `{context}` 자리에 검색해서
정리한 글이 들어가고, `{question}` 에 질문이 들어갑니다.

In [12]:
#프롬프트 라이브러리 임포트 
from langchain_core.prompts import ChatPromptTemplate

#"당신은 위니브 카페 상담원입니다. " "아래 [참고자료]만 보고 답하세요. 프롬프트 작성하세요
rag_prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        "당신은 위니브 카페 상담원입니다."
        "아래 [참고자료]만 보고 답하세요.\n\n"
        "[참고자료]\n {context}"
    ),
    (
        "human", "{question}"
    ),
])
    
print("프롬프트 준비 완료 ✅")

프롬프트 준비 완료 ✅


**이 셀이 한 일**: 검색 결과(`{context}`)와 질문(`{question}`)을
받는 프롬프트를 만들었습니다.

### 셀 7 — 검색 → 프롬프트 → AI 연결 (오늘의 핵심!)

`|` 로 부품을 잇습니다. 핵심은 `{"context": 검색기 | 정리,
"question": 질문그대로}` — 이게 1단계 그림의 ②③④ 를 한 번에
표현합니다.

In [13]:
from langchain_openai import ChatOpenAI
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

# context : 질문으로 검색하고 → 정리   (retriever | format_docs)
# question: 질문을 그대로 전달         (RunnablePassthrough)
rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | rag_prompt
    | llm
    | StrOutputParser()
)

# 질문("환불은 며칠 안에 신청해야 하나요?")만 넣으면 (검색 → 정리 → AI → 답) 이 한 번에!
print(rag_chain.invoke("환불은 며칠 안에 신청해야 하나요?"))

환불은 수령 후 7일 이내에 신청하셔야 합니다.


**이 셀이 한 일**: 질문 하나만 넣었는데 검색부터 답변까지
자동으로 흘렀습니다. **이게 RAG 체인입니다!**

> 💡 `RunnablePassthrough()` = "질문을 손 안 대고 그대로 다음으로
> 넘겨라" 라는 뜻입니다. 검색엔 질문으로 검색하고, 프롬프트엔
> 원래 질문을 그대로 줘야 하니까요.

# 4️⃣ 단계 — 위니브 RAG 챗봇 완성

이제 진짜 위니브 문서로, **출처 붙고 거짓말 안 하는** 진짜 RAG
챗봇을 완성합니다.

### 셀 8 — 위니브 문서 → 벡터 DB 

"불러오기 → 쪼개기 → 분류 → 벡터 DB"를
한 셀에 모았습니다. 

In [31]:
import os
from langchain_community.document_loaders import DirectoryLoader, TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

#1. Documents 객체 생성   documents 담아(저장)주세요
documents = {
    "01_faq.md":        "# 위니브 카페 자주 묻는 질문 (FAQ)\n\n스페셜티 커피 전문 쇼핑몰 위니브 카페 고객센터입니다.\n자주 묻는 질문을 모았습니다.\n\n---\n\n### Q. 주문은 어떻게 하나요?\nA. 원하시는 상품을 장바구니에 담은 뒤 결제 페이지에서 배송지와 결제수단을\n입력하시면 됩니다. 회원·비회원 모두 주문 가능합니다.\n\n### Q. 비회원도 주문할 수 있나요?\nA. 네, 비회원 주문이 가능합니다. 다만 비회원 주문은 적립금이 적립되지 않으며,\n회원 등급 혜택도 적용되지 않습니다.\n\n### Q. 주문을 취소하고 싶어요.\nA. 상품이 발송되기 전이라면 마이페이지 > 주문내역에서 직접 취소할 수 있습니다.\n이미 발송된 경우에는 반품 절차로 진행됩니다. 자세한 내용은 환불·반품 정책을\n확인해 주세요.\n\n### Q. 결제 수단은 무엇이 있나요?\nA. 신용·체크카드, 계좌이체, 간편결제(카카오페이·네이버페이)를 지원합니다.\n무통장입금은 현재 지원하지 않습니다.\n\n### Q. 선물 포장이 되나요?\nA. 네, 가능합니다. 주문 시 '선물포장' 옵션을 선택하시면 건당 2,000원에\n포장해 드립니다. 선물 메시지 카드도 함께 넣어드립니다.\n\n### Q. 주문한 상품은 언제 받을 수 있나요?\nA. 평일 오후 2시 이전 결제 건은 당일 발송되며, 보통 발송 후 1~2일 이내\n도착합니다. 지역에 따라 더 걸릴 수 있습니다. 배송 정책 문서를 참고해 주세요.\n\n### Q. 배송 조회는 어디서 하나요?\nA. 마이페이지 > 주문내역에서 송장번호와 배송 상태를 확인할 수 있습니다.\n발송 완료 시 알림 문자도 발송됩니다.\n\n### Q. 원두를 분쇄해서 보내주나요?\nA. 네, 주문 시 분쇄도(홀빈·드립·에스프레소 등)를 선택하실 수 있습니다.\n신선도를 위해 홀빈(분쇄 안 함) 구매를 권장드립니다.\n\n### Q. 재고가 없는 상품은 언제 다시 들어오나요?\nA. 품절 상품 페이지에서 '재입고 알림'을 신청하시면 입고 시 문자로\n안내드립니다. 원두는 로스팅 일정에 따라 보통 주 1회 입고됩니다.\n\n### Q. 영수증·세금계산서 발급이 되나요?\nA. 현금영수증은 결제 시 신청 가능하며, 세금계산서는 사업자 회원에 한해\n고객센터로 요청하시면 발급해 드립니다.\n\n### Q. 상품 문의는 어디로 하나요?\nA. 상품 상세페이지의 Q&A 게시판 또는 고객센터 채팅으로 문의해 주세요.\n운영 시간은 평일 오전 10시~오후 5시입니다.",
    "02_refund.md":     "# 위니브 카페 환불·반품·교환 정책\n\n본 정책은 전자상거래법을 따르며, 위니브 카페에서 구매한 모든 상품에 적용됩니다.\n\n## 1. 반품·환불 신청 기간\n\n- **단순 변심**: 상품 수령일로부터 7일 이내 신청 가능\n- **상품 하자·오배송**: 수령일로부터 14일 이내 신청 가능\n- 위 기간이 지나면 반품·환불이 제한될 수 있습니다.\n\n## 2. 환불 불가 품목 (중요)\n\n다음 품목은 단순 변심에 의한 반품·환불이 **불가**합니다.\n\n- **개봉한 원두 및 드립백**: 식품 위생상 개봉 시 재판매가 불가능합니다.\n  미개봉 상태에서는 7일 이내 반품 가능합니다.\n- **디저트류(케이크·쿠키·구움과자 등)**: 신선식품으로, 단순 변심 반품이\n  불가합니다. 단, 변질·파손 등 하자가 있는 경우 수령 14일 이내 교환·환불됩니다.\n- **주문 제작 굿즈(각인·커스텀 텀블러 등)**: 고객 요청으로 제작된 상품은\n  단순 변심 반품이 불가합니다.\n- 단, 위 품목도 **상품 하자나 오배송**인 경우에는 환불·교환이 가능합니다.\n\n## 3. 단순 변심 반품 시 비용\n\n- 단순 변심으로 반품할 경우 왕복 배송비(편도 3,000원 × 2 = 6,000원)는\n  고객이 부담합니다.\n- 상품 하자·오배송으로 인한 반품은 배송비를 위니브 카페가 전액 부담합니다.\n\n## 4. 환불 처리 절차 및 기간\n\n1. 마이페이지 > 주문내역에서 반품 신청\n2. 반품 상품 발송 (회수 택배 신청 또는 직접 발송)\n3. 반품 상품 도착 후 상태 확인\n4. 확인 완료 후 **영업일 기준 3일 이내** 환불 처리\n\n- 카드 결제는 카드사 정책에 따라 취소 반영까지 영업일 3~5일이 추가로\n  소요될 수 있습니다.\n- 적립금으로 결제한 부분은 적립금으로 환원됩니다.\n\n## 5. 교환\n\n- 동일 상품의 색상·옵션 변경은 재고가 있는 경우 교환 가능합니다.\n- 하자 상품의 교환 배송비는 위니브 카페가 부담합니다.\n- 단순 변심에 의한 교환 배송비는 고객이 부담합니다.\n- 디저트류는 교환이 어려워 환불로 처리될 수 있습니다.\n\n## 6. 주문 취소\n\n- 상품 발송 **전**에는 마이페이지에서 즉시 취소 가능하며 전액 환불됩니다.\n- 발송 **후**에는 취소가 불가하며, 수령 후 반품 절차로 진행됩니다.\n\n## 7. 환불이 제한되는 경우\n\n- 고객의 사용·훼손으로 상품 가치가 현저히 감소한 경우\n- 개봉 후 상태인 식품(원두·드립백·디저트)의 단순 변심\n- 반품 기간(7일/14일)을 경과한 경우\n- 위 사유라도 상품 자체 하자가 입증되면 환불 대상이 됩니다.",
    "03_shipping.md":   "# 위니브 카페 배송 정책\n\n## 1. 배송비\n\n| 구분 | 배송비 |\n|------|--------|\n| 기본 배송비 | 3,000원 |\n| 5만원 이상 구매 | 무료 |\n| 제주 지역 | 기본 배송비 + 3,000원 |\n| 도서산간 지역 | 기본 배송비 + 3,000원 |\n\n- 5만원 이상 구매 시 기본 배송비는 무료이나, 제주·도서산간 추가비\n  (3,000원)는 별도로 부과됩니다.\n\n## 2. 배송 기간\n\n- 평일 **오후 2시 이전** 결제 완료 건은 당일 발송됩니다.\n- 오후 2시 이후 결제 건은 다음 영업일에 발송됩니다.\n- 발송 후 도착까지 보통 1~2일 소요됩니다.\n- 제주·도서산간은 2~4일이 소요될 수 있습니다.\n- 주말·공휴일은 발송되지 않습니다.\n\n## 3. 원두 로스팅 일정\n\n- 원두는 신선도를 위해 **주문 후 로스팅(주문 로스팅)** 으로 발송됩니다.\n- 로스팅은 매주 화·금요일에 진행되며, 해당 요일 직전 주문이 함께\n  발송됩니다. 이로 인해 원두 단일 주문은 최대 3~4일이 소요될 수 있습니다.\n- 원두와 일반 상품을 함께 주문하면 로스팅 일정에 맞춰 합포장 발송됩니다.\n\n## 4. 배송 조회\n\n- 마이페이지 > 주문내역에서 송장번호로 실시간 조회가 가능합니다.\n- 발송 완료 시 등록된 휴대폰으로 알림 문자가 발송됩니다.\n\n## 5. 배송 관련 안내\n\n- 택배사는 일반 택배(CJ대한통운)를 이용합니다.\n- **새벽배송·당일배송 서비스는 운영하지 않습니다.**\n- 부재 시 택배사 정책에 따라 경비실 위탁 또는 재배송됩니다.\n- 배송지 오기입으로 인한 반송 시 재발송 배송비는 고객이 부담합니다.\n\n## 6. 해외 배송\n\n- 현재 해외 배송은 지원하지 않습니다. 국내 배송만 가능합니다.",
    "04_catalog.md":    "# 위니브 카페 상품 정보 카탈로그\n\n각 상품에는 `카테고리` 정보가 있습니다. (메타데이터 필터 실습에 사용)\n카테고리: 원두 / 드립백 / 디저트 / 홈카페기기 / 바리스타굿즈\n\n---\n\n## [원두] 에티오피아 예가체프 G1\n\n- 카테고리: 원두\n- 가격: 18,000원 (250g)\n- 설명: 자스민·베르가못 향이 도는 밝은 산미의 워시드 원두입니다.\n- 보관: 직사광선을 피해 서늘한 곳에 보관, 개봉 후 2주 이내 소비 권장\n- 환불: 미개봉 7일 이내 가능 / 개봉 시 환불 불가\n\n## [원두] 브라질 산토스 NY2\n\n- 카테고리: 원두\n- 가격: 14,000원 (250g)\n- 설명: 견과류·다크초콜릿의 묵직한 바디. 에스프레소 베이스로 인기.\n- 보관: 밀폐 용기 보관, 개봉 후 2주 이내 소비 권장\n- 환불: 미개봉 7일 이내 가능 / 개봉 시 환불 불가\n\n## [드립백] 데일리 블렌드 드립백 10개입\n\n- 카테고리: 드립백\n- 가격: 9,000원 (10개입)\n- 설명: 사무실·여행용 휴대 간편 드립백. 균형 잡힌 블렌드.\n- 보관: 개별 포장, 실온 보관\n- 환불: 미개봉 7일 이내 가능 / 개봉 시 환불 불가\n\n## [디저트] 클래식 파운드 케이크\n\n- 카테고리: 디저트\n- 가격: 12,000원\n- 설명: 버터 풍미가 진한 수제 파운드 케이크. 커피와 페어링 추천.\n- 보관: 냉장 보관, 수령 후 3일 이내 섭취 권장\n- 환불: 신선식품으로 단순 변심 반품 불가 / 변질·파손 시 14일 이내 교환·환불\n\n## [디저트] 비스코티 세트 (6개)\n\n- 카테고리: 디저트\n- 가격: 9,500원\n- 설명: 아몬드·헤이즐넛 두 가지 맛 구움과자 세트.\n- 보관: 실온 보관, 수령 후 7일 이내 섭취 권장\n- 환불: 신선식품으로 단순 변심 반품 불가 / 하자 시 14일 이내 교환·환불\n\n## [홈카페기기] 핸드드립 세트 (드리퍼+서버)\n\n- 카테고리: 홈카페기기\n- 가격: 38,000원\n- 설명: 입문용 드리퍼와 600ml 유리 서버 구성.\n- 보관: -\n- 환불: 미사용 7일 이내 가능 / 하자 시 14일 이내 교환\n\n## [홈카페기기] 전동 그라인더 GR-200\n\n- 카테고리: 홈카페기기\n- 가격: 89,000원\n- 설명: 40단계 분쇄도 조절 코니컬 버 그라인더.\n- 보관: -\n- 환불: 미사용 7일 이내 가능 / 하자(작동 불량) 시 14일 이내 무상 교환\n\n## [바리스타굿즈] 위니브 로고 머그컵\n\n- 카테고리: 바리스타굿즈\n- 가격: 12,000원\n- 설명: 300ml 도자기 머그. 위니브 카페 로고 인쇄.\n- 보관: -\n- 환불: 미사용 7일 이내 가능\n\n## [바리스타굿즈] 커스텀 각인 텀블러 (주문제작)\n\n- 카테고리: 바리스타굿즈\n- 가격: 28,000원\n- 설명: 원하는 문구를 각인해 제작하는 보온 텀블러.\n- 보관: -\n- 환불: **주문 제작 상품으로 단순 변심 반품 불가** / 하자 시 교환 가능",
    "05_membership.md": "# 위니브 카페 멤버십·적립 안내\n\n## 1. 회원 등급\n\n회원 등급은 **최근 6개월 누적 구매액**을 기준으로 매월 1일 자동 산정됩니다.\n\n| 등급 | 최근 6개월 누적 구매액 | 적립률 |\n|------|----------------------|--------|\n| 베이직 | 0원 ~ | 1% |\n| 실버 | 10만원 이상 | 2% |\n| 골드 | 30만원 이상 | 3% |\n| VIP | 60만원 이상 | 5% |\n\n- 등급은 매월 1일 재산정되며, 기준 미달 시 하향될 수 있습니다.\n- 신규 가입 시 기본 등급은 베이직입니다.\n\n## 2. 적립금\n\n- 구매 확정 시 등급별 적립률에 따라 적립됩니다.\n- 적립금은 **적립일로부터 1년** 경과 시 자동 소멸됩니다.\n- 적립금은 1,000원 이상부터 사용 가능하며, 1원 단위로 사용할 수 있습니다.\n- 비회원 주문은 적립이 적용되지 않습니다.\n\n## 3. 신규 가입 혜택\n\n- 신규 회원가입 시 3,000원 웰컴 적립금이 즉시 지급됩니다.\n- 웰컴 적립금의 유효기간은 지급일로부터 30일입니다.\n\n## 4. 생일 쿠폰\n\n- 생일이 등록된 회원에게 생일月 1일에 10% 할인 쿠폰이 발급됩니다.\n- 생일 쿠폰은 발급일로부터 그 달 말일까지 사용 가능합니다.\n- 일부 할인 상품에는 중복 적용되지 않습니다.\n\n## 5. 적립금 환불 처리\n\n- 적립금으로 결제한 주문을 환불할 경우, 사용한 적립금은 적립금으로\n  환원됩니다(현금 환불 아님).\n- 환원된 적립금의 유효기간은 원래 적립금의 잔여 기간을 따릅니다.\n\n## 6. 등급 혜택 요약\n\n- 골드 등급 이상: 무료 선물포장 제공\n- VIP 등급: 신상품 우선 구매 및 분기별 시음 원두 증정",
}
# os 패키지에서 제공하는 makedirs() 디렉토리 chat_docs 생성하고 각 파일을 생성하여 저장하세요
os.makedirs("chat_docs", exist_ok=True)
for  filename, content in documents.items():
    with open(f"chat_docs/{filename}","w",encoding="utf-8") as f:
        f.write(content)

all_docs = DirectoryLoader(
       "chat_docs",glob="*.md",loader_cls=TextLoader,loader_kwargs={"encoding":"utf-8"}
).load()

chunks = RecursiveCharacterTextSplitter(chunk_size=300 , chunk_overlap=30).split_documents(all_docs)

winib_db = Chroma.from_documents(chunks,embeddings,collection_name="winib")
print("위니브 벡터 DB 준비 완료 ✅  청크 수:",winib_db._collection.count())

위니브 벡터 DB 준비 완료 ✅  청크 수: 75


**이 셀이 한 일**: 위니브 문서 5개를 청크로 만들어 벡터 DB에
저장했습니다. 

### 셀 9 — 위니브 검색기 + 출처 정리 함수

셀 4·5에서 배운 그대로. 위니브 벡터 DB에 검색 창구를 달고,
파일명을 출처로 보기 좋게 정리합니다.

In [32]:
winib_retriever = winib_db.as_retriever(search_kwargs={'k':3})

def winib_format(docs):
    blocks = []
    for doc in docs:
        filename = os.path.basename(doc.metadata["source"])
        blocks.append(f"[출처 : {filename}]\n{doc.page_content}")
    return "\n\n".join(blocks)

# 잘 "배송비 얼마예요?" 되는지 질문으로 미리 확인
preview = winib_format(winib_retriever.invoke("배송비 얼마에요?"))
print(preview)

[출처 : 03_shipping.md]
# 위니브 카페 배송 정책

## 1. 배송비

| 구분 | 배송비 |
|------|--------|
| 기본 배송비 | 3,000원 |
| 5만원 이상 구매 | 무료 |
| 제주 지역 | 기본 배송비 + 3,000원 |
| 도서산간 지역 | 기본 배송비 + 3,000원 |

- 5만원 이상 구매 시 기본 배송비는 무료이나, 제주·도서산간 추가비
  (3,000원)는 별도로 부과됩니다.

## 2. 배송 기간

[출처 : 03_shipping.md]
# 위니브 카페 배송 정책

## 1. 배송비

| 구분 | 배송비 |
|------|--------|
| 기본 배송비 | 3,000원 |
| 5만원 이상 구매 | 무료 |
| 제주 지역 | 기본 배송비 + 3,000원 |
| 도서산간 지역 | 기본 배송비 + 3,000원 |

- 5만원 이상 구매 시 기본 배송비는 무료이나, 제주·도서산간 추가비
  (3,000원)는 별도로 부과됩니다.

## 2. 배송 기간

[출처 : 03_shipping.md]
# 위니브 카페 배송 정책

## 1. 배송비

| 구분 | 배송비 |
|------|--------|
| 기본 배송비 | 3,000원 |
| 5만원 이상 구매 | 무료 |
| 제주 지역 | 기본 배송비 + 3,000원 |
| 도서산간 지역 | 기본 배송비 + 3,000원 |

- 5만원 이상 구매 시 기본 배송비는 무료이나, 제주·도서산간 추가비
  (3,000원)는 별도로 부과됩니다.

## 2. 배송 기간


**이 셀이 한 일**: 위니브 검색기를 만들고, 검색 결과를
`[출처: 03_shipping.md]` 처럼 출처가 붙은 글로 정리했습니다.

### 셀 10 — 위니브 RAG 챗봇 완성 (출처 + 거짓말 방지)

1단계에서 말한 두 가지 예의를 프롬프트에 넣습니다:
**① 자료에 없으면 모른다고 ② 답 끝에 출처 적기**

In [33]:
winib_prompt = ChatPromptTemplate.from_messages([
    ("system",
     "당신은 위니브 카페 고객센터 상담원입니다. "
     "아래 [참고자료]만 보고 정중히 답하세요. "
     "참고자료에 없는 내용은 지어내지 말고 "
     "'문서에서 확인되지 않습니다'라고 답하세요. "
     "답 끝에 사용한 출처를 (출처: 파일명) 형태로 적으세요.\n\n"
     "[참고자료]\n{context}"),
    ("human", "{question}"),
])

winib_chatbot = (
    {"context": winib_retriever | winib_format, "question": RunnablePassthrough()}
    | winib_prompt
    | llm
    | StrOutputParser()
)

print(winib_chatbot.invoke("개봉한 원두를 단순 변심으로 환불할 수 있나요?"))

개봉한 원두는 식품 위생상 재판매가 불가능하므로, 단순 변심으로 환불이 불가능합니다. 미개봉 상태에서는 7일 이내에 반품이 가능합니다. (출처: 02_refund.md)


**이 셀이 한 일**: 위니브 RAG 챗봇을 완성하고 질문했습니다.

> ✅ 답이 "개봉 원두는 단순변심 환불 **불가**" + `(출처:
> 02_refund.md)` 형태면 성공입니다. **진짜 RAG 챗봇 완성!**
> 1교시에서 AI가 지어냈던 그 질문을, 이제 정확히 + 출처까지 답합니다.

### 셀 11 — 여러 질문으로 출처 확인

주제가 다른 질문 3개를 던져, 출처가 알맞게 붙는지 봅니다.

In [34]:
questions = [
    "VIP 등급 적립률이 몇 %인가요?",
    "제주도 배송비는 어떻게 되나요?",
    "케이크 단순변심으로 반품돼요?",
]
# 아래 출력예시를 보고 출력문 작성하세요

for q in questions:
    print(winib_chatbot.invoke(q))

VIP 등급의 적립률은 5%입니다. (출처: 05_membership.md)
문서에서 확인되지 않습니다. (출처: 03_shipping.md)
케이크를 단순 변심으로 반품하실 수 있습니다. 단, 이 경우 왕복 배송비 6,000원(편도 3,000원 × 2)은 고객님께서 부담하셔야 합니다. (출처: 02_refund.md)


**이 셀이 한 일**: 주제별 질문에 답하게 했습니다.

> 💡 등급 질문엔 (출처: 05_membership.md), 배송엔
> (출처: 03_shipping.md) 처럼 **질문에 맞는 출처**가 붙으면
> 검색·연결이 잘 된 것입니다.

### 셀 12 — 거짓말 방지 확인 (없는 질문)

위니브 문서에 **없는** 내용을 물어봅니다. 좋은 챗봇은 지어내지
않고 "확인되지 않습니다"라고 해야 합니다.

In [35]:
no_info_questions = [
    "위니브 카페 매장에 주차장이 있나요?",
    "와이파이 비밀번호 알려주세요",
]

# 아래 출력예시를 보고 출력문 작성하세요

for q in questions:
    print("Q: ", q)
    print("A: ", winib_chatbot.invoke(q))

Q:  VIP 등급 적립률이 몇 %인가요?
A:  VIP 등급의 적립률은 5%입니다. (출처: 05_membership.md)
Q:  제주도 배송비는 어떻게 되나요?
A:  문서에서 확인되지 않습니다. (출처: 03_shipping.md)
Q:  케이크 단순변심으로 반품돼요?
A:  케이크를 단순 변심으로 반품하실 수 있습니다. 단, 이 경우 왕복 배송비 6,000원(편도 3,000원 × 2)은 고객님께서 부담하셔야 합니다. (출처: 02_refund.md)


**이 셀이 한 일**: 문서에 없는 질문을 던졌습니다.

> ✅ 두 답 모두 "문서에서 확인되지 않습니다" 류면 성공입니다.
> 모르는 걸 지어내지 않는 것 — 고객센터에서 이게 신뢰의 핵심입니다.
> ('거짓말'을 제대로 막았습니다.)

---
# 🎓 배운 것 정리

| 단계 | 한 일 | 핵심 |
|------|-------|------|
| 1️⃣ 이론 | 부품 조립 + 검색기 개념 | 서고(DB) vs 사서(검색기) |
| 2️⃣ 기본 | as_retriever / format_docs | 검색 결과를 한 글로 |
| 3️⃣ 응용 | 검색 → AI 한 줄 연결 | RAG 체인 |
| 4️⃣ 실습 | 위니브 RAG 완성 | 출처 + 거짓말 방지 ✅ |

**한 문장 요약**: 검색기로 관련 청크를 자동으로 찾아 AI에게 주고,
출처를 붙이고, 없으면 모른다고 답하게 한다. = 진짜 RAG 챗봇.

**다음 시간(5교시)**: 검색이 가끔 엉뚱한 청크를 가져옵니다.
검색을 더 똑똑하게 만드는 법을 배웁니다.

---

